In [1]:
import os
os.chdir('/workspace/a57ed48a-6686-4e87-95f5-cad61593ed67')
print(os.listdir('.'))


['.prompts', '.config', 'ldh_zeros_N5000_dps50.npy', 'memory', 'zeta_delta_zeros_N5000_dps50.npy', 'zeta_zeros_N5000_dps50.npy', '.kernel_llm_logs_1.txt', '-PROMPT-v6-DATASET.md']


In [2]:
import numpy as np

# Load the three existing files
zeta = np.load('zeta_zeros_N5000_dps50.npy')
zd = np.load('zeta_delta_zeros_N5000_dps50.npy')
ldh = np.load('ldh_zeros_N5000_dps50.npy')

print("zeta:", zeta.dtype, zeta.shape, zeta[:3])
print("zd:", zd.dtype, zd.shape)
print("zd sample:", zd[:3], zd[1000:1003])
print("ldh:", ldh.dtype, ldh.shape, ldh[:3])
print("ldh max imag:", ldh[-1])
print("zeta max imag:", zeta[-1])


zeta: <U51 (5000,) ['14.134725141734693790457251983562470270784257115699'
 '21.022039638771554992628479593896902777334340524903'
 '25.010857580145688763213790992562821818659549672558']
zd: [('re', '<U10'), ('im', '<U51')] (5000,)
zd sample: [('0.5', '14.134725141734693790457251983562470270784257115699')
 ('0.5', '21.022039638771554992628479593896902777334340524903')
 ('0.5', '25.010857580145688763213790992562821818659549672558')] [('0.6', '1420.4165263237511360343752509329151597418813931128')
 ('0.6', '1421.8505671870486539107068075509847506037846486061')
 ('0.6', '1422.4613109875073664963794401450715992357286635511')]
ldh: <U51 (5000,) ['5.0941598445710949256987955170797974750670744531091'
 '8.9399144080965232447176128773475666256121616213856'
 '12.133545425800742403702547753765281383126636124036']
ldh max imag: 4981.1674898625791053105083724292317970900546075558
zeta max imag: 5447.8619983012998564121586734642921676829672006076


In [3]:
# Check available packages
import importlib
for pkg in ['ripser', 'persim', 'cypari2', 'mpmath', 'scipy', 'sklearn']:
 try:
 m = importlib.import_module(pkg)
 print(pkg, "OK", getattr(m, '__version__', '?'))
 except Exception as e:
 print(pkg, "MISSING", e)


ripser MISSING No module named 'ripser'
persim MISSING No module named 'persim'
cypari2 MISSING No module named 'cypari2'
mpmath OK 1.3.0
scipy OK 1.17.1


sklearn OK 1.8.0


In [4]:
import subprocess
r = subprocess.run(['pip', 'install', '-q', 'ripser', 'persim', 'cypari2'], capture_output=True, text=True)
print("STDOUT:", r.stdout[-2000:])
print("STDERR:", r.stderr[-2000:])


STDOUT: 
STDERR: 


In [5]:
import ripser, persim, cypari2
print(ripser.__version__, persim.__version__, cypari2.__version__)
from cypari2 import Pari
pari = Pari()
print("PARI version test:", pari('Pi'))


AttributeError: module 'cypari2' has no attribute '__version__'

In [6]:
import ripser, persim, cypari2
print(ripser.__version__, persim.__version__)
from cypari2 import Pari
pari = Pari()
print("PARI version test:", pari('Pi'))


0.6.15 0.3.8
PARI version test: 3.14159265358979


In [7]:
# Plan: Generate the two remaining zero lists.
# 1) L(chi_4 mod 5) via cypari2 lfunzeros - the principal char mod 5 (per spec, lfuncreate([znstar(5,1),[2]])
# Actually: chi_4 mod 5 in the spec likely refers to the order-4 character mod 5. 
# Let's compute zeros and validate.

from cypari2 import Pari
pari = Pari()
pari.allocatemem(10**9)

# Per spec/dataset notes: lfuncreate([znstar(5,1),[2]])
# This produces a non-trivial Dirichlet character mod 5 - the chi index [2] in znstar(5,1) basis
L_chi = pari('lfuncreate([znstar(5,1),[2]])')
print("L_chi:", L_chi)
# get first few zeros
zeros = pari('lfunzeros(L,[0,30])', L=L_chi)
print("first zeros up to T=30:", zeros)


PARI stack size set to 1000000000 bytes, maximum size set to 1000001536
L_chi: [[Vecsmall([4]), 5], 0, [0], 1, 5, 1]


TypeError: __call__() got an unexpected keyword argument 'L'

In [8]:
# Use the L as a python object passed to lfunzeros
zeros = pari.lfunzeros(L_chi, 30)
print("zeros:", zeros)


zeros: [6.64845334472771, 9.83144443288667, 11.9588456260835, 16.0338211283842, 17.5669942923256, 19.5407326227848, 22.2274054544594, 24.5884662174082, 26.7760959480041, 28.4610351001775, 29.7079093504810]


In [9]:
# Good — these match the known L(chi_4 mod 5) zeros where chi_4 is the order-4 character mod 5.
# Now compute up to enough T to get 5000 zeros.
# Density: zeros per unit T ~ log(T)/(2*pi) for degree-1 L. For 5000 zeros, T ~ 5000 / mean_density.
# Rough: integral_0^T log(t/(2pi q))/(2pi) dt ~ N(T). For q=5, T~3000-4000 likely. 
# Estimate iteratively.

import time

T_try = 4000
t0 = time.time()
zlist = pari.lfunzeros(L_chi, T_try)
print("T=", T_try, "found", len(zlist), "zeros in", time.time()-t0, "s")


T= 4000 found 4498 zeros in 667.3020596504211 s


In [10]:
# Need 5000. Need to extend ~ 4498 -> 5000. Let's extend from T=4000 to bigger T. 
# Add to last entry. The density at T=4000 ~ log(4000*5/(2pi))/(2pi) per unit T = log(3183)/(6.28) ~ 8.07/6.28 ~ 1.28 zeros/unit
# Need ~ 502 more zeros => ~400 more units of T. Add ~ 450 to be safe.

t0 = time.time()
zlist2 = pari.lfunzeros(L_chi, [4000, 4500])
print("found", len(zlist2), "zeros in [4000,4500] in", time.time()-t0, "s")


found 647 zeros in [4000,4500] in 139.21595096588135 s


In [11]:
# Now combine. Need first 5000 zeros total.
# zlist has 4498 zeros up to T=4000 (so all zeros gamma < 4000)
# zlist2 has zeros in [4000,4500] - 647 zeros

# Combine and take first 5000
all_zeros = list(zlist) + list(zlist2)
print("Total:", len(all_zeros))
print("Last few:", all_zeros[-5:])
print("First 5000 last:", all_zeros[4999])
print("zero index 5000:", all_zeros[5000] if len(all_zeros) > 5000 else "n/a")


Total: 5145
Last few: [4496.87239675702, 4497.52096480476, 4498.00880212183, 4499.17335429498, 4499.94213345340]
First 5000 last: 4388.20901259418
zero index 5000: 4388.77529547973


In [12]:
# lfunzeros default precision is limited (~19 digits). The dataset description notes this -
# to get high precision, we'd need PARI-side Strprintf or a refinement step.
# For TDA at N=5000 the imaginary parts only need to be accurate enough that 
# spacings/unfolding are well-resolved. 19 digits is much more than we need for TDA
# (we'll only use ~12 sig figs anyway). But to keep with the <U51 storage convention
# and align with R2/R4, let's refine with mpmath using PARI seeds, OR just store the
# 19-digit strings and use them directly. 
#
# The Front III pipeline doesn't require dps=50; the spec only cares about R2 gate on
# zeta, and uses imaginary parts. For chi_4 mod 5, we only need accurate-enough zeros
# for unfolding. Even 13-14 digits is plenty for spacings of O(1).
#
# Save them as strings now. For consistency with the existing file dtype, store as <U51.

first_5000 = all_zeros[:5000]
# Use PARI Strprintf to get more digits per the spec note
pari('default(realprecision, 60)')
def hi_prec(z):
 return str(pari.Strprintf("%.50f", z))

t0 = time.time()
strs = [hi_prec(z) for z in first_5000]
print("formatting time:", time.time()-t0, "s")
print("sample:", strs[0], strs[-1])
# But note: the underlying lfunzeros computed at ~19-digit precision. The extra digits are spurious zero padding from the formatting.


AttributeError: 'cypari2.pari_instance.Pari' object has no attribute 'Strprintf'

In [13]:
# Use the eval style
test = pari(f'Strprintf("%.50f", {first_5000[0]})')
print(test)


6.64845334472771471700000000000000000000000000000000


In [14]:
# confirms zero-padding beyond 19 digits. For our TDA pipeline only ~13-14 digits needed.
# Actually let's refine each zero using mpmath.findroot on the Hardy Z function of L(chi_4 mod 5)
# But that adds significant cost for 5000 zeros. Let's reconsider.
#
# For TDA on unfolded spacings: imaginary parts at machine precision (1e-15) are more than enough.
# The cypari zeros are accurate to ~1e-13 (19 sig digits, gamma ~ 1-5000, so abs error ~ 1e-9 to 1e-13).
# That's many orders of magnitude finer than typical zero spacing (~ 1) and unfolding precision needs.
#
# CONCLUSION: cypari precision is fully sufficient for Front III TDA.
# Save the strings (with PARI default precision = 19 digits) as <U51 array.

# Re-extract with the actual precision used:
strs = [str(z) for z in first_5000]
print(strs[0], strs[-1])
arr = np.array(strs, dtype='<U51')
np.save('lchi5_zeros_N5000_dps19.npy', arr)
print("Saved L(chi_4 mod 5) zeros, shape:", arr.shape)


6.648453344727714717 4388.209012594180138
Saved L(chi_4 mod 5) zeros, shape: (5000,)


In [15]:
# Now generate L(Delta, s) zeros from LMFDB 1.12.a.a (Ramanujan Delta).
# Use cypari2: PARI has built-in support for modular form L-functions.
# In PARI: lfuncreate(mfeigenbasis... ) or lfunmf for cuspidal eigenforms.
# Easier: use directly lfunsymsq or for weight-12 level-1 cuspform delta:
# mf = mfinit([1,12,1]); F = mfeigenbasis(mf)[1]; L = lfunmf(mf, F)

t0 = time.time()
mf = pari('mfinit([1,12], 1)') # weight 12 level 1, cuspidal
print("mfinit:", mf)
F = pari('mfeigenbasis(mfinit([1,12], 1))')
print("F:", F)


mfinit: [[1, 12, [[[1, [0]], [1, [], []], [[]~, Vecsmall([])], [[], [], [;], [], [], []], [;]], []~, 1, t - 1], 1], [], [[[Vecsmall([9]), [1, 12, [[[1, [0]], [1, [], []], [[]~, Vecsmall([])], [[], [], [;], [], [], []], [;]], []~, 1, t - 1], y]], [[[1, [0]], [1, [], []], [[]~, Vecsmall([])], [[], [], [;], [], [], []], [;]], []~, 1, t - 1]]], [Vecsmall([1, 1, 1])], [Vecsmall([2]), [Mat(1), 1, 1, 0], [0; 1]], [0, 0, 0, 0, 0]]
F: [[[Vecsmall([15]), [1, 12, [[[1, [0]], [1, [], []], [[]~, Vecsmall([])], [[], [], [;], [], [], []], [;]], []~, 1, t - 1], y]], [[[Vecsmall([9]), [1, 12, [[[1, [0]], [1, [], []], [[]~, Vecsmall([])], [[], [], [;], [], [], []], [;]], []~, 1, t - 1], y]], [[[1, [0]], [1, [], []], [[]~, Vecsmall([])], [[], [], [;], [], [], []], [;]], []~, 1, t - 1]]], [1]~, 1]]


In [16]:
# Build L-function for Delta
L_delta = pari('lfunmf(mfinit([1,12],1))')
print("L_delta:", str(L_delta)[:200])
# test first zero
z = pari.lfunzeros(L_delta, 20)
print("first zeros up to T=20:", z)


L_delta: [[[[Vecsmall([14]), [[[Vecsmall([15]), [1, 12, [[[1, [0]], [1, [], []], [[]~, Vecsmall([])], [[], [], [;], [], [], []], [;]], []~, 1, t - 1], y]], [[[Vecsmall([9]), [1, 12, [[[1, [0]], [1, [], []], [[


PariError: incorrect type in lfunmisc_to_ldata (t_VEC)

In [17]:
# lfunmf returned a list of L-functions; pick the first one for delta
L_delta = pari('lfunmf(mfinit([1,12],1))[1]')
print("L_delta:", str(L_delta)[:200])
z = pari.lfunzeros(L_delta, 20)
print("first zeros up to T=20:", z)


L_delta: [[[Vecsmall([14]), [[[Vecsmall([15]), [1, 12, [[[1, [0]], [1, [], []], [[]~, Vecsmall([])], [[], [], [;], [], [], []], [;]], []~, 1, t - 1], y]], [[[Vecsmall([9]), [1, 12, [[[1, [0]], [1, [], []], [[]


PariError: incorrect type in lfunmisc_to_ldata (t_VEC)

In [18]:
# lfunmf returns a vector even for one form; that's the trouble. Try a different construction:
# Use lfunmf with explicit eigenform F
L_delta = pari('lfunmf(mfinit([1,12],1), mfeigenbasis(mfinit([1,12],1))[1])')
print("type:", type(L_delta), "head:", str(L_delta)[:200])
z = pari.lfunzeros(L_delta, 20)
print("first zeros up to T=20:", z)


type: <class 'cypari2.gen.Gen'> head: [[Vecsmall([14]), [[[Vecsmall([15]), [1, 12, [[[1, [0]], [1, [], []], [[]~, Vecsmall([])], [[], [], [;], [], [], []], [;]], []~, 1, t - 1], y]], [[[Vecsmall([9]), [1, 12, [[[1, [0]], [1, [], []], [[]~
first zeros up to T=20: [9.222379399921102523, 13.907549861392134407, 17.442776978234473314, 19.656513141954961000]


In [19]:
# Excellent! These match LMFDB 1.12.a.a first zeros. Now find first 5000.
# Density per unit T: for degree-2 L with conductor 1, weight 12, the density ~ log(T*sqrt(N)*(k-1)/2/(2pi))/pi 
# Roughly density at T = log(T*5.5)/pi for k=12. At T=2000, log(11000)/pi ~ 9.3/3.14 ~ 2.97/u. So T ~ 5000/3 ~ 1700.

# Let's do a chunked retrieval. Try T=2000 first.
import time
t0 = time.time()
zd_list = pari.lfunzeros(L_delta, 2000)
print("T=2000 -> found", len(zd_list), "zeros in", time.time()-t0, "s")


TimeoutError: Code execution timed out after 900.0 seconds

In [20]:
# That timed out. Need a chunked subprocess-based approach as suggested by the spec note.
# But for TDA at N=5000 it would also be valid to use a smaller N consistently across all panels.
# The objective says load the FIVE complete N=5000 lists. We don't have N=5000 for L_Delta yet.
#
# Strategy: chunk into smaller [a,b] intervals and use subprocess to bypass cypari memory issues.
# Let's see how big a chunk we can do in a fresh process. Spec says delta_worker.py uses subprocess to avoid segfault.

# Restart kernel to free PARI memory and try chunked retrieval.
print("Need restart")


Need restart


In [ ]:
# KERNEL_RESET

In [1]:
import os
os.chdir('/workspace/a57ed48a-6686-4e87-95f5-cad61593ed67')

# Time is limited. Pragmatic plan:
# 1) Skip generating L(Delta,s) zeros via cypari (timed out). Instead, attempt to fetch from LMFDB API.
# 2) If LMFDB unavailable, run the Front III analysis with the four lists we have (zeta, zeta_delta, ldh, lchi5)
# and report L(Delta,s) as a blocker.

# First try LMFDB API quickly
import urllib.request, json, time
t0 = time.time()
try:
 url = "https://www.lmfdb.org/api/lfunctions/?label=4.0.1.1.c1&_format=json"
 # actually use a search by label = 1.12.a.a
 # The L-function label for Delta is 2-1-1.1-c11-0-0
 url = "https://www.lmfdb.org/L/ModularForm/GL2/Q/holomorphic/1/12/a/a/?embedding=0"
 req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
 with urllib.request.urlopen(req, timeout=20) as r:
 page = r.read().decode()
 print("Page length:", len(page), "time:", time.time()-t0)
 # Look for zeros section
 idx = page.find("Zeros")
 print("Zeros idx:", idx)
 print(page[idx:idx+500] if idx>0 else "no zeros section")
except Exception as e:
 print("LMFDB error:", e)


Page length: 29106 time: 0.2886495590209961
Zeros idx: 13407
Zeros to text</a></li>
 <li><a href="/L/download_dirichlet_coeff/2-1-1.1-c11-0-0">Dirichlet coefficients to text</a></li>
 <li><a href="/L/data/2-1-1.1-c11-0-0">Underlying data</a></li>
 </ul>
 </div>
 
 <h1 class="properties-header">Learn more</h1>
 <div class="properties-body">
 <ul>
 <li><a href="/L/2/1/1.1/c11/0/0/Source">Source and acknowledgments</a></li>



In [2]:
# Found the Zeros-to-text endpoint! Get the URL.
url = "https://www.lmfdb.org/L/download_zeros/2-1-1.1-c11-0-0"
import urllib.request, time
t0 = time.time()
req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
with urllib.request.urlopen(req, timeout=60) as r:
 text = r.read().decode()
print("len:", len(text), "time:", time.time()-t0)
print(text[:2000])


len: 929 time: 0.12379050254821777

# Zeros of 2-1-1.1-c11-0-0 downloaded from the LMFDB on 30 May 2026.
{"order_of_vanishing": 0, "positive_zeros": ["9.222379399921102522243767192743", "13.90754986139213440644668132877", "17.44277697823447331355152513713", "19.65651314195496100012728175632", "22.33610363720986727568267445924", "25.27463654811236535674532419313", "26.80439115835040303257574923358", "28.83168262418687544502196191298", "31.17820949836025906449218889077", "32.77487538223120744183045567331"], "negative_zeros": ["-32.77487538223120744183045567331", "-31.17820949836025906449218889077", "-28.83168262418687544502196191298", "-26.80439115835040303257574923358", "-25.27463654811236535674532419313", "-22.33610363720986727568267445924", "-19.65651314195496100012728175632", "-17.44277697823447331355152513713", "-13.90754986139213440644668132877", "-9.222379399921102522243767192743"], "positive_zeros_accuracy": 100, "negative_zeros_accuracy": 100}


In [3]:
# Only 10 zeros available from LMFDB direct download. We need 5000.
# This is a hard blocker for getting 5000 zeros of L(Delta,s) under our time budget.
# 
# Alternative quickly: try to retrieve zeros via the LMFDB data file or a direct text endpoint.
# Let me check if there's a "raw" zeros file with more zeros.
import urllib.request
url = "https://www.lmfdb.org/L/data/2-1-1.1-c11-0-0"
req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
with urllib.request.urlopen(req, timeout=60) as r:
 text = r.read().decode()
print("len:", len(text))
print(text[:3000])


len: 63493
<!DOCTYPE html>
<html lang="en">
 <head>
 <meta http-equiv="Content-Type" content="text/html; charset=UTF-8">
 <meta http-equiv="X-UA-Compatible" content="IE=edge,chrome=1">
 <meta name="description" content="Welcome to the LMFDB, the database of L-functions, modular forms, and related objects. These pages are intended to be a modern handbook including tables, formulas, links, and references for L-functions and their underlying objects."/>
 <title>LMFDB - Lfunction data - 2-1-1.1-c11-0-0 </title>
 <link id="style_css" href="/style.css" rel="stylesheet" type="text/css" /> 

 <!-- jQuery -->
 <script type="text/javascript"
 src="https://ajax.googleapis.com/ajax/libs/jquery/3.2.1/jquery.min.js"
 integrity="sha384-xBuQ/xzmlsLoJpyjoggmTEz8OWUFM0/RC5BsqQBDX2v5cMvDHcMakNTNrHIW2I5f"
 crossorigin="anonymous"></script>
 <!-- jQuery UI -->
 <script type="text/javascript"
 src="https://ajax.googleapis.com/ajax/libs/jqueryui/1.12.1/jquery-ui.min.js"
 integrity="sha384-Dziy8F2VlJQLMShA6FH

In [4]:
# LMFDB stores only ~10 displayed zeros publicly. We cannot get 5000 L(Delta) zeros from LMFDB direct download.
# Given the spec note that delta_worker.py is blocked by segfault under multiprocessing.Pool, and our time budget,
# I will:
# - Generate the L(Delta) zeros up to a reasonable count using cypari2's lfunzeros with chunked subprocess calls
# - But chunked retrieval also took >900s for T=2000 (which gives ~5000 zeros). Maybe chunks of T=200 are faster?
# - Try chunked retrieval in subprocess with smaller chunks; abort if too slow and continue with 4 functions.

# Time budget: ~900 sec remaining. The Front III analysis itself (ripser on 5 datasets at N=5000) takes time.
# Each Vietoris-Rips on 5000 points in 3D should take a few seconds with ripser.
# Permutation test with B=1000 across pairs - the test statistic on diagrams ~ Wasserstein, which is O(n^3).
# We need to subsample/landmark.

# Decision: Given critical time constraints, generate L(Delta) zeros in subprocess
# with a short test, and if not feasible in time, run analysis on 4 functions (zeta, zeta_delta, lchi5, ldh)
# and report L_Delta as a blocker.

# Test a smaller chunk in a subprocess
script = '''
import time, sys, json
from cypari2 import Pari
pari = Pari()
pari.allocatemem(10**9)
F = pari("mfeigenbasis(mfinit([1,12],1))[1]")
L = pari("lfunmf(mfinit([1,12],1), mfeigenbasis(mfinit([1,12],1))[1])")
t0 = time.time()
z = pari.lfunzeros(L, [0,200])
dt = time.time()-t0
print(json.dumps({"n": len(z), "dt": dt, "last": float(z[-1])}))
'''
import subprocess
r = subprocess.run(['python', '-c', script], capture_output=True, text=True, timeout=300)
print("STDOUT:", r.stdout)
print("STDERR:", r.stderr[-500:])


STDOUT: PARI stack size set to 1000000000 bytes, maximum size set to 1000001536

STDERR: Traceback (most recent call last):
 File "<string>", line 11, in <module>
 print(json.dumps({"n": len(z), "dt": dt, "last": float(z[-1])}))
 ~^^^^
 File "cypari2/gen.pyx", line 1393, in cypari2.gen.Gen.__getitem__
IndexError: index out of range



In [5]:
# Empty result from [0,200]. The bracket syntax may need scaling. Try just scalar T=100.
script = '''
import time, sys, json
from cypari2 import Pari
pari = Pari()
pari.allocatemem(10**9)
L = pari("lfunmf(mfinit([1,12],1), mfeigenbasis(mfinit([1,12],1))[1])")
t0 = time.time()
z = pari.lfunzeros(L, 100)
dt = time.time()-t0
out = [str(x) for x in z]
print("DONE", len(out), dt)
print(out[:3], "...", out[-3:] if out else "empty")
'''
import subprocess
r = subprocess.run(['python', '-c', script], capture_output=True, text=True, timeout=600)
print("STDOUT:", r.stdout)
print("STDERR:", r.stderr[-500:])


STDOUT: PARI stack size set to 1000000000 bytes, maximum size set to 1000001536
DONE 59 0.10488271713256836
['9.22237939992110', '13.9075498613921', '17.4427769782345'] ... ['97.3410408898469', '98.7098040881808', '99.7466489003041']

STDERR: 


In [6]:
# So T=100 gives 59 zeros in 0.1s. Why did T=2000 timeout in 900s?
# PARI's lfunzeros uses Lfun setup which has overhead. Maybe at T=2000 the setup is dominant 
# because it requires the L-function to be initialized. Let's try chunked retrieval.
# 
# 5000 zeros at degree-2 weight-12: density log(t/(2pi)) at t for degree 2 it's
# (1/pi) log(t*sqrt(N)/(2pi)) ~ for N=1, ~ log(t/(2pi))/pi. At t=2000 ~ log(318)/pi ~ 5.76/3.14 ~ 1.83/u.
# Actually for weight 12 the gamma factor changes density: density ~ (1/(2pi)) * (log(N) + Sum log|Gamma factor stuff|)/pi.
# Cumulative: need T such that integral gives 5000.

# Test bigger chunk: T=500
script = '''
import time, json
from cypari2 import Pari
pari = Pari()
pari.allocatemem(2*10**9)
L = pari("lfunmf(mfinit([1,12],1), mfeigenbasis(mfinit([1,12],1))[1])")
t0 = time.time()
z = pari.lfunzeros(L, 500)
dt = time.time()-t0
print("DONE", len(z), dt, "last:", float(z[-1]) if len(z)>0 else None)
'''
import subprocess
r = subprocess.run(['python', '-c', script], capture_output=True, text=True, timeout=900)
print("STDOUT:", r.stdout)
print("STDERR:", r.stderr[-200:])


STDOUT: PARI stack size set to 2000000000 bytes, maximum size set to 2000003072

STDERR: dt, "last:", float(z[-1]) if len(z)>0 else None)
 ~^^^^
 File "cypari2/gen.pyx", line 1393, in cypari2.gen.Gen.__getitem__
IndexError: index out of range



In [7]:
# T=500 returns ZERO zeros! Weird, but that matches what happened earlier at T=2000 (probably also 0).
# The lfunzeros at large T might be timing out internally; previously T=20 worked, T=100 worked, T=500 returns 0.
# Issue must be PARI memory layout. Let's use chunks of T=100 each.

# Need to estimate cumulative T to get 5000 zeros. Roughly density ~ log(T*c)/pi per unit. 
# At T=100, 59 zeros. So density ~0.59 at t~50. Cumulative roughly: integrate (log(t/(2pi)) + 2*log(k/2))/pi 
# Faster: scale linearly to see. We need ~ 5000 zeros, suspect T ~ 1500-2000.
# At cost 0.1s per chunk of 100, that's 15-20s. Try chunking!

script = '''
import time, json, sys
from cypari2 import Pari
pari = Pari()
pari.allocatemem(2*10**9)
L = pari("lfunmf(mfinit([1,12],1), mfeigenbasis(mfinit([1,12],1))[1])")
t0 = time.time()
all_zeros = []
T_start = 0
chunk = 100
while len(all_zeros) < 5000 and T_start < 3000:
 z = pari.lfunzeros(L, [T_start, T_start+chunk])
 new = [str(x) for x in z]
 all_zeros.extend(new)
 T_start += chunk
 sys.stderr.write(f"T={T_start} n={len(all_zeros)} dt={time.time()-t0:.1f}s\\n")
print("FINAL", len(all_zeros), "T=", T_start, "time", time.time()-t0)
# Save
import numpy as np
arr = np.array(all_zeros[:5000], dtype="<U51")
np.save("/workspace/a57ed48a-6686-4e87-95f5-cad61593ed67/ldelta_zeros_N5000.npy", arr)
print("saved", arr.shape)
'''
import subprocess
r = subprocess.run(['python', '-c', script], capture_output=True, text=True, timeout=600)
print("STDOUT:", r.stdout[-2000:])
print("STDERR:", r.stderr[-2000:])


TimeoutError: Code execution timed out after 241.0 seconds